In [3]:
# Project: BPO Operations Analytics
# Description: Generational scripts with snake_case practice

from pyspark.sql import functions as F
from pyspark.sql.types import *
import random

# 1. Path setup (Medallion)
path_genesys = "Files/bronze/genesys"
path_csat = "Files/bronze/csat"
path_employees = "Files/bronze/employees"

# 2. Employees list generation
print ("invoking employees list...")

# 2.1. Lists to generate employees with real names
names = ["Andres", "Camila", "Carlos", "Daniela", "Diego", "Elena", "Felipe", "Gabriela", "Javier", "Laura", "Luis", "Maria", "Nicolas", "Paula", "Ricardo", "Sofia", "Valentina", "Sebastian", "Isabella", "Mateo"]
last_names = ["Garcia", "Martinez", "Lopez", "Rodriguez", "Perez", "Sanchez", "Gomez", "Diaz", "Torres", "Ramirez", "Hernandez", "Ruiz", "Morales", "Castillo", "Jimenez", "Moreno", "Castro", "Ortiz", "Alvarez", "Gutiérrez"]

# 2.2 Agent's list generation
    # random.sample with names and last_names to add diversity

agents = [
    (
        f"A00{i}",
        f"{random.choice(names)} {random.choice(last_names)}",
        random.choice(["Bogota", "Medellin", "Chia"]),
        random.choice(["Alma Gomez", "Luis Montenegro"])
    )
    for i in range(1, 51)
]

# 2.3 Schema definition
emp_schema = StructType([
    StructField("agent_id", StringType(), False),
    StructField("agent_name", StringType(), False),
    StructField("site_location", StringType(), False),
    StructField("supervisor", StringType(), False)
])

df_employees = spark.createDataFrame(agents, emp_schema)

#2.4 Write in path_employees/agent_directory.csv
df_employees.write.mode("overwrite").csv(f"{path_employees}/agent_directory.csv", header=True)

# 3. Interaction generation
print("Invoking interactions...")

df_interactions = spark.range(0, 5000).withColumn("interaction_id", F.expr("uuid()")) \
    .withColumn("agent_id", F.expr("concat('A00', cast(rand() * 50 + 1 as int))")) \
    .withColumn("timestamp", F.expr("date_add(current_timestamp(), -cast(rand() * 7 as int))")) \
    .withColumn("talk_time_sec", F.expr("cast(rand() * 600 as int)")) \
    .withColumn("hold_time_sec", F.expr("cast(rand() * 120 as int)")) \
    .withColumn("acw_time_sec", F.expr("cast(rand() * 60 as int)")) \
    .withColumn("queue_name", F.expr("element_at(array('CustomerCare', 'TechnicalSupport', 'Sales'), cast(rand()*3+1 as int))"))

# 3.1 Inject NULLS
df_interactions = df_interactions.withColumn("talk_time_sec", 
    F.when(F.rand() > 0.95, F.lit(None)).otherwise(F.col("talk_time_sec")))

#3.2 Write in path_employees/agent_directory.csv
df_interactions.write.mode("overwrite").csv(f"{path_genesys}/calls_log.csv", header=True)

#4. CSAT generation
print("Invoking CSAT...")

df_csat = df_interactions.sample(0.3).select(
    "interaction_id",
    F.expr("cast(rand() * 5 + 1 as int)").alias("survey_score")
)
df_csat.write.mode("overwrite").csv(f"{path_csat}/surveys.csv", header=True)

print("Behold, \n An offering")



StatementMeta(, 5aa0bc1f-9930-4df2-a496-bf6576a00dc0, 5, Finished, Available, Finished)

invoking employees list...
Invoking interactions...
Invoking CSAT...
Behold, 
 An offering
